In [1]:
import numpy as np
import pandas as pd

In [2]:
activity = pd.read_csv('datasets/movie-metrics/activity.csv', parse_dates=['date'])
users = pd.read_csv('datasets/movie-metrics/users.csv', parse_dates=['created_at'])

# Method 1: proper and simplistic version

In [3]:
started_finished = (
    activity
    .groupby('user_id')['finished']
    .agg(['count', 'sum'])
    .rename(columns={
        'count': 'started',
        'sum': 'finished'
    })
)

In [4]:
first_movies = activity[activity['finished'] == 1].groupby('user_id').first() \
    .rename(columns={'movie_name': 'first_movie', 'date': 'first_date'}) \
    .drop(columns=['finished', 'id'])
last_movies = activity[activity['finished'] == 1].groupby('user_id').last() \
    .rename(columns={'movie_name': 'last_movie', 'date': 'last_date'}) \
    .drop(columns=['finished', 'id'])

In [5]:
# Most of these groupby queries return with index so that pd.concat matches without many params
# This might be getting overwhelmed for beginners if they have not glanced over the whole documentation
user_summary = pd.concat([first_movies, last_movies, started_finished], axis='columns').reset_index()

In [6]:
user_summary

,user_id,first_date,first_movie,last_date,last_movie,started,finished
0,1,2023-09-12,Turning Red,2025-03-26,Her,30,26
1,2,2023-06-22,The Shawshank Redemption,2025-05-01,Fight Club,15,12
2,3,2023-11-10,Oppenheimer,2025-03-31,Nope,10,7
3,4,2023-07-27,Fight Club,2025-05-09,Avengers: Endgame,43,34
4,5,2023-09-07,Top Gun: Maverick,2025-03-14,Bohemian Rhapsody,31,25
5,6,2023-12-18,Inception,2025-03-27,The Revenant,13,11
6,7,2024-08-28,Avengers: Endgame,2024-09-15,Soul,2,2
7,8,2024-01-24,Arrival,2025-04-25,Whiplash,29,23
8,9,2024-02-25,Big Hero 6,2025-04-11,Moonlight,17,12
9,10,2024-05-01,Whiplash,2025-03-08,Knives Out,7,6


## How many users have "Fight Club" as the last film they've seen?

In [7]:
user_summary.query('last_movie == "Fight Club"')

,user_id,first_date,first_movie,last_date,last_movie,started,finished
1,2,2023-06-22,The Shawshank Redemption,2025-05-01,Fight Club,15,12
17,18,2024-10-01,Fight Club,2025-05-16,Fight Club,35,28
19,20,2024-11-27,Luca,2025-05-02,Fight Club,34,19


# Method 2: returns multiple values with the same date

In [8]:
num_finished = (
    activity
    .groupby('user_id')['finished']
    .agg(['count', 'sum'])
    .rename(columns={
        'count': 'started',
        'sum': 'finished'
    })
)

In [9]:
first_and_last = (
    activity[activity['finished'] == 1]
    .groupby('user_id')['date']
    .agg(['min','max'])
    .rename(columns={
        'min':'first_date',
        'max': 'last_date'
    })
)

In [10]:
user_summary = (
    pd.concat([first_and_last, num_finished], axis='columns')
    .reset_index()
    .merge(activity, left_on=['user_id', 'first_date'], right_on=['user_id', 'date'])
    .merge(activity, left_on=['user_id', 'last_date'], right_on=['user_id', 'date'])
)
user_summary = user_summary[[
    'user_id',
    'first_date', 'movie_name_x',
    'last_date', 'movie_name_y',
    'started', 'finished_x'
]].rename(columns={
    'movie_name_x': 'first_movie',
    'movie_name_y': 'last_movie',
    'finished_x': 'finished'
})

In [11]:
user_summary

,user_id,first_date,first_movie,last_date,last_movie,started,finished
0,1,2023-09-12,Turning Red,2025-03-26,Her,30,26
1,2,2023-06-22,The Shawshank Redemption,2025-05-01,Fight Club,15,12
2,3,2023-11-10,Oppenheimer,2025-03-31,Nope,10,7
3,4,2023-07-27,Fight Club,2025-05-09,Avengers: Endgame,43,34
4,5,2023-09-07,Top Gun: Maverick,2025-03-14,Bohemian Rhapsody,31,25
5,6,2023-12-18,Inception,2025-03-27,The Revenant,13,11
6,7,2024-08-28,Avengers: Endgame,2024-09-15,Soul,2,2
7,8,2024-01-24,Arrival,2025-04-25,Whiplash,29,23
8,9,2024-02-25,Big Hero 6,2025-04-11,Moonlight,17,12
9,10,2024-05-01,Whiplash,2025-03-08,Knives Out,7,6


In [12]:
user_summary.query('last_movie == "Fight Club"')

,user_id,first_date,first_movie,last_date,last_movie,started,finished
1,2,2023-06-22,The Shawshank Redemption,2025-05-01,Fight Club,15,12
19,18,2024-10-01,Fight Club,2025-05-16,Fight Club,35,28
21,20,2024-11-27,Luca,2025-05-02,Fight Club,34,19
22,20,2024-11-27,The Wolf of Wall Street,2025-05-02,Fight Club,34,19
